# Diffie-Hellman Key Exchange vs ML-KEM

In [ ]:
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import dh, mlkem
from cryptography.hazmat.primitives.kdf import hkdf

from cryptography.hazmat.primitives.serialization import (
    Encoding, ParameterFormat, PublicFormat
)

import binascii
import base64

## DHKE

I'm Alice now.
Alice generates the parameters ($p$, $g$, $q$) and sends them to Bob.



In [ ]:
parameters = dh.generate_parameters(generator=2, key_size=1024)

In [ ]:
# Copy the output string and send it to Bob
parameters.parameter_bytes(Encoding.PEM,ParameterFormat.PKCS3)

I'm still Alice. I need a local keypair.

In [ ]:
# Generate Alice's keypair
alice_private_key = parameters.generate_private_key()
alice_public_key = alice_private_key.public_key()

# Copy the output string and send it to Bob
alice_public_key.public_bytes(Encoding.PEM,PublicFormat.SubjectPublicKeyInfo)

I'm Bob now

In [ ]:
# uncomment the following line and paste the parameter string that you received from Alice
# parameters = serialization.load_pem_parameters(b'...parameter string...')

In [ ]:
# uncomment the following line and paste the key received from Alice
# alice_public_key = serialization.load_pem_public_key(b'...alice public key...')

# generate Bob's keypair
bob_private_key = parameters.generate_private_key()
bob_public_key = bob_private_key.public_key()

# This is the shared group element
bob_shared_secret = bob_private_key.exchange(alice_public_key)

I'm Alice again

In [ ]:
# uncomment the following line and paste the key received from Bob
# bob_public_key = serialization.load_pem_public_key(b'...bob public key...")

# This is the shared group element
alice_shared_secret = alice_private_key.exchange(bob_public_key)

I'm either Alice or Bob now.

In [ ]:
if alice_shared_secret == bob_shared_secret:
  print("OK")
else:
  print("KO")

In [ ]:
# The secret is not a key, you must derive a key using HKDF
derived_key = hkdf.HKDF(
    algorithm=hashes.SHA256(),
    length=32,
    salt=None,
    info=b'some shared data',
).derive(alice_shared_secret)
binascii.hexlify(derived_key)

## ML-KEM

I'm Alice

In [ ]:
alice_private_key = mlkem.MLKEM768PrivateKey.generate()
alice_public_key = alice_private_key.public_key()
alice_public_key.public_bytes(Encoding.PEM,PublicFormat.SubjectPublicKeyInfo)

I'm Bob

In [ ]:
# uncomment and paste
# alice_public_key = mlkem.MLKEM1024PublicKey.from_public_bytes(public_bytes)
# alice_public_key = serialization.load_pem_public_key("...string...")

alice_public_key = serialization.load_pem_public_key(b'-----BEGIN PUBLIC KEY-----\nMIIEsjALBglghkgBZQMEBAIDggShAHdCiq2xS8JRV+djisTLNGXJCM9xJG2RlBuZ\nEY7rdvz4DalkycjUbPBUzC/2o6z3WhTWtCaRunUaHqIlW3d3TjkDKueFkZFMFugV\ndolQzR1ISA4WKQbKMUdake2AgUdRiHHTAyoTympJD6OZI4ewTeGrLcJlB8OJVQTE\nR2VINuxLEq3MggsqZw5mtcnMSlR2fLzDdcRFHgqlgJPotQkMaVsBKCFmCOmIEXeZ\nNRh0TceSj5RmWl7WMY4ZGShyF9SWrhULB9HUICygBSNMjrUKuiWAvS9qExoLeJ1Y\nu7LwXTbVcrpoRyxczTiMYYQ7TMr1HYdgSIUMF4qqx3lmOX3bJDL1YEKWnXxrwufT\nhpB0CI36qeIAO0LnbeeMm2lStoDcAYQqG6BqFPliK4Vkeo1ll1fEXoPUe/BJoTDZ\nrNAgudWqV7j1AV/KXTghGCxRsGhBJvR2m5TVFxY6fmC3MVsQDJCkoF0nfpzRjJms\nTfOGvVv8uAxqsd4IaCYzhYiTIYohp7q7uaJ8SmWYbFQ0rIcVTPl8wMlEEBwlnmry\nUlfEWF+JUeySp9+ZEie8FG3Vqb6kSxhThOBYuSf4oMhBCMuFRbbxgHh5fp7ToYO8\nrY9mwbv6hibKAx0QCA82MYn8kkQsI8jZxcaaC4BWjUjkm2XbsQD9jQJ0OnCTAowj\nXevMK/52fnUKJqfyKqLIEwLDSLTIU6cMvz06wqUFHoKaeE6zWqJQSuYQmrqHJdH3\nGxh7tmhymJI4HLsoWsPRUc7Xu4pyJYIxtEnQqrhRly3QnF88wW12hEkHtji8ZHvI\nYknJWP77mpj1xRoYMP+VBQFBpWpMuLKFQ++xBXjwnDkEsRhLedzDOvILUtJDbBqs\nsSeiIMkJFZO8XBj1RQH2PDIZZV2xaQDxTHj8QxmVhG5lUbzXqf9jgmW1PxE2Iip5\nAiyLcQijhRy8A1WkUoWczUcxhn+6em2sxLIErG1aatPwLazSEPNYddFTIWsBH+QV\ntbwomMBqYMwzBOWIp+68GntUTOuhDppDoBXReW/VlQsbF8VhAV1keISQbxY1He5L\nUWUQjbqxXnbyw1HHVXwncF7nI+YsyGXlnBbrq9XVtzYBC/pDzBB7z2wCJDMCUJLz\nIHj1ls+aOXrzvt7EmSdyLIKbrV1mVccFZwMnY31DSlkMHSV4azjVthbkPeyIOaG5\nXNGkT9GLO7+gSBrUzY/aBr6wDm51LUTVNYvlN73pfQAVCLVXOfGlXV+nCY8Qc03r\nDdxElVOBPUrkI0kjGx4rdswaktZ2pX5WABXJZaaJVfBkos9CCTEIc8sFYsumxkWI\nx+O3xd+Zx5pgnY7gHQg1VLeJoczrUcnKMVFhjHCUfbfVFD14pp7SQvyFkUJ7E3CG\nqhv5Em+0yqOQCETGfheEElU1lRLBbwwYw+XAQwlhyKrVbir6n4PEY7thfqujS8mq\nL2/hWBbLkTNZllqGUcmHMkAmiq6IbydIgRurvhTHVAEFtw0cWXwCLy3rjeE6Wcc3\nJdmsKTQIuUrlXLoFyNzTG7dqC8lRucso3t+Z+v1mVEFjJEh3xjbcKUA7uIgopM6q\naRFKA44/\n-----END PUBLIC KEY-----\n')

bob_shared_secret, ciphertext = alice_public_key.encapsulate()

# There is no standard way to convert the ciphertext (a string of bytes) to printable characters.
# We can use Base64
encoded_ciphertext = base64.b64encode(ciphertext).decode('ascii')
encoded_ciphertext

I'm Alice again

In [ ]:
# Recover the Base64 ciphertext
# ciphertext = base64.b64decode(b'...ciphertext...', validate=True)

alice_shared_secret = alice_private_key.decapsulate(ciphertext)

In [ ]:
if alice_shared_secret == bob_shared_secret:
  print("OK")
else:
  print("KO")

# The ML-KEM secret could be used as a key, however it is suggested to use a KDF

## Laboratory

Pick a classmate, decide who's Alice and who's Bob. Exchange a key using DHKE.
You can use email or chat to exchange the protocol messages. Measure the sizes of the messages in both directions and verify that the exchanged key is correct.

Then exchange a key using ML-KEM.